In [1]:
!apt-get update -q
!apt-get install -y wget tar

# Скачиваем и устанавливаем NVIDIA HPC SDK
!curl https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
!sudo apt-get update -y
!sudo apt-get install -y nvhpc-24-3-cuda-multi

# Добавим в PATH
import os
os.environ["PATH"] += ":/opt/nvidia/hpc_sdk/Linux_x86_64/24.3/compilers/bin"
!which nvc

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,372 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,287 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:14

In [2]:
!which nvc

/opt/nvidia/hpc_sdk/Linux_x86_64/24.3/compilers/bin/nvc


In [4]:
%%writefile pi_openacc.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <sys/time.h>

double wtime() {
    struct timeval tv;
    gettimeofday(&tv, NULL);
    return tv.tv_sec + tv.tv_usec * 1e-6;
}

double calc_pi_openacc(long n) {
    double sum = 0.0;
    double coef = 1.0 / (double)n;
    #pragma acc data copyout(sum)
    {
        #pragma acc parallel loop reduction(+:sum)
        for (long i = 0; i < n; ++i) {
            double xi = (i + 0.5) * coef;
            sum += 4.0 / (1.0 + xi * xi);
        }
    }
    return sum * coef;
}

int main(int argc, char **argv) {
    if (argc < 2) {
        printf("Usage: %s <n_iterations>\n", argv[0]);
        return 1;
    }
    long n = atol(argv[1]);
    if (n <= 0) n = 100000000;

    double pi_ref = 0.0;
    double coef = 1.0 / (double)n;
    double t0 = wtime();
    for (long i = 0; i < n; ++i) {
        double xi = (i + 0.5) * coef;
        pi_ref += 4.0 / (1.0 + xi * xi);
    }
    double t1 = wtime();
    pi_ref *= coef;

    printf("CPU pi = %.12f, time = %.6f s\n", pi_ref, t1 - t0);

    double tg0 = wtime();
    double pi_gpu = calc_pi_openacc(n);
    double tg1 = wtime();
    printf("OpenACC pi = %.12f, time = %.6f s\n", pi_gpu, tg1 - tg0);
    printf("Abs error = %.12e\n", fabs(pi_gpu - pi_ref));
    return 0;
}


Writing pi_openacc.c


In [5]:
!nvc -acc -Minfo=accel -O2 pi_openacc.c -o pi_openacc
!./pi_openacc 100000000

calc_pi_openacc:
     16, Generating copyout(sum) [if not already present]
         Generating implicit firstprivate(i,n)
         Generating NVIDIA GPU code
         18, #pragma acc loop gang, vector(128) /* blockIdx.x threadIdx.x */
             Generating reduction(+:sum)
     18, Generating implicit firstprivate(coef)
CPU pi = 3.141592653590, time = 0.070624 s
OpenACC pi = 3.141592653590, time = 0.169985 s
Abs error = 4.236611061970e-13
